<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Train and Evaluate your Classification Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_3_classification_part_2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

3-4 hours of activities and 1-3 hours of waiting for models to train

In this second lab, you will train and evaluate your classification model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Given that the steps for fine-tuning a classification model do not vary too much, you may be able to run a lot of the existing code without changing it too much. However, some steps may still not be applicable to your project, and likewise, you may have to complete additional steps, so treat the template in this lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.

## Overview

In this second part of implementing your capstone project, you will focus on fine-tuning and evaluating your text classification model.

### What you will learn

By the end of this second part of the capstone project, you will be able to:

* Fine-tune a large language model using LoRA for classification tasks.

* Evaluate a classification model's prediction accuracy and generalization.

### Tasks

In this lab, you will:

* Load the preprocessed dataset that you prepared in the previous lab.

* Configure the Gemma model and LoRA hyperparameters.

* Execute the fine-tuning process.

* Test the fine-tuned model and run manual spot checks on unseen data.

* Iterate on the hyperparameter settings and fine-tuning process until you are satisfied with your model's behavior.

The end result of this lab will be a classification model that can categorize your examples as you specified in your task description.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Discover the Transformer Architecture).

- **Fine-tuning models using LoRA** (Courses 05 Fine-tune Your Model and 07 Accelerate Your Model).  


**This lab needs to be run on GPU. Choose a T4 GPU.** See the section "How to use Google Colaboratory (Colab)" below for instructions on how to do this.

## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are executed on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports

The following cell contains imports for a range of libraries that you may use for fine-tuning and evaluating your model. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import random # For random number generation and shuffling of data.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill  # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

import matplotlib.pyplot as plt # Plotting and visualisation.

# Classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report

# Set Keras and JAX settings.
os.environ["KERAS_BACKEND"] = "jax"

# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

# Deep learning and AI libraries.
import jax.numpy as jnp  # JAX numerical computing.
import keras  # High-level neural networks API.
import keras_nlp  # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.

##  Activity 1: Load your data

As a first step, you will load your training and test data that you prepared in the previous activity from Google Drive. See the worked example for how to do this.

In [ ]:
# Connect this notebook to Google Drive.
drive.mount('/content/drive')

#### Worked example: Load data

```python
train_df = pd.read_json(
    "/content/drive/MyDrive/classification_training_data.jsonl", lines=True
)

val_df = pd.read_json(
    "/content/drive/MyDrive/classification_validation_data.jsonl", lines=True
)

test_df = pd.read_json(
    "/content/drive/MyDrive/classification_test_data.jsonl", lines=True
)

# Turn dataframe into a list of dictionaries, as expected by the model
# fine-tuning code.
train_data = train_df.to_dict('records')
val_data = val_df.to_dict('records')
test_data = test_df.to_dict('records')

# Print the first element in train_data, val_data, test_data for verification.
print(train_data[0])
print(val_data[0])
print(test_data[0])
```

#### Your implementation

In [ ]:
# Add your code for loading your data here.


### Configure your Kaggle API key for Gemma

To access Gemma models, provide your Kaggle credentials. If you have not set your Kaggle API key yet take a look at the instruction in the following cells.

#### Set up a Kaggle account



To run this notebook, you will have to sign up for [Kaggle](https://www.kaggle.com), a platform that hosts datasets and models for machine learning, and sign the agreement for using the Gemma 3 model. This is required so that you can download the weights of the Gemma model for fine-tuning.

**Step 1: Create your Kaggle account**

* Go to the Kaggle website: https://www.kaggle.com

* Click the "Register" button in the top-right corner.

* You can sign up using your Google account (recommended for easy Colab integration) or by entering an email and password.

* Follow the on-screen prompts to complete your registration and verify your email.

**Step 2: Sign the Gemma 3 model agreement**

* Make sure you are logged into your new Kaggle account.

* Go directly to the Gemma 3 model card page: https://www.kaggle.com/models/keras/gemma3/keras/

* You should see a "Request Access" button.

* Click the button, read through the license agreement, and click "Accept" to gain access to the model. You must do this before the API will let you download the model.

**Step 3: Generate your Kaggle API key**

* From any Kaggle page, click on your profile picture or icon in the top-right corner.

* Select "Account" from the drop-down menu.

* Scroll down to the "API" section.

* Click the "Create Legacy API Key" button in the "Legacy API Credentials" section.

* This will immediately download a file named `kaggle.json` to your computer. This file contains your username and your secret API key. Keep it safe.

**Step 4: Set your API Key in Colab**

* Click the "key" icon 🔑 in the left-hand sidebar.

* You will see the "Secrets" panel.

* Now, open the kaggle.json file you downloaded on your computer. It's a simple text file and will look like this:

   ```json
   {"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_KAGGLE_API_KEY"}
   ```
* In the Colab Secrets panel, create two new secrets:

   1. Name: `KAGGLE_USERNAME`

      Value: Copy and paste `YOUR_KAGGLE_USERNAME` from your `kaggle.json` file.

   2. Name: `KAGGLE_KEY`

      Value: Copy and paste `YOUR_KAGGLE_API_KEY` from your `kaggle.json` file.

* For both secrets, make sure the "Notebook access" toggle is switched on.


#### Load the Kaggle username and key

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

## Activity 2: Preparation for fine-tuning


Before starting fine-tuning, ensure that your training data is formatted correctly for the specific model you plan to use. In the previous notebook, you created JSONL files containing classification examples, which can be reused across different models and fine-tuning methods.

However, each model has its own formatting requirements. The Gemma models support only two conversational roles, **user** and **model**, but not a **system** role. In other words, prompts and responses are structured as direct exchanges between the user and the model, without a separate system message to provide global instructions or context (such as defining tone, style, or constraints). Higher-capacity LLMs, like Gemini, also use a system role for this purpose, allowing developers to set overall behavior at the start of a conversation. Because Gemma does not support the system role, each training example must explicitly include any necessary context or instruction within the user prompt itself. For more information, see the [Gemma formatting and system instructions document](https://ai.google.dev/gemma/docs/core/prompt-structure/).
  


------
> 💻 **Your task:**
>
> 1. **Implement conversation formatting**:
>    - **Special tokens**: The Gemma model uses `<start_of_turn>` and `<end_of_turn>`.
>    - **Role labels**: The Gemma model uses `user` and `model` roles.
>    - **Turn structure**: Adapt the `format_io_as_turns()` function in the worked example to ensure it matches your conversation format requirements.
>
> 2. **Verify formatted output**:
>    - Write code to examine examples to verify:
>      - User turns are properly formatted with the prompt for classification.
>      - Model turns contain the expected classification responses.
>      - Special tokens are correctly positioned.
>      - The final data dictionary has the required "prompts" and "responses" keys.
>
> 3. **Customize for your domain**:
>    - Ensure the formatting aligns with how you want the model to behave during inference.
>

------

#### Worked example: Add special tokens



The code below transforms data that you prepared in the previous notebook into a conversational format suitable for fine-tuning a Gemma model.

The goal is to convert this into a training dataset containing user prompts and model responses with the classification:

| Field | Purpose |
|-------|--------|
| **`input`** | The actual text the model will classify (e.g.,  the description). |
| **`output`** | The correct label(s) the model should produce for this record. |

---

**Single-task classification**

If you only need **category classification**, a record could look similar to the following one:

```json
{
  "input": "<start_of_turn>user\nClassify the following item by category: Jollof Rice.<end_of_turn>",
  "output": "<start_of_turn>model\nFood<end_of_turn>"
}
```

**Multi-task classification**

If you wish to predict more than one task in one pass, then you need a **multi-task classification**, and the instructions and output must be adapted accordingly. For example:

```json
{
  "input": "<start_of_turn>user\nClassify the following item by category and region: Jollof Rice.<end_of_turn>",
  "output": "<start_of_turn>model\n{\"category\": \"Food\", \"region\": \"West Africa\"}<end_of_turn>"
}
```

In the multi-task classification example above, the instruction asks for two fields: category and region. The output is a JSON-formatted string with both values.

The training data are structured with special tokens that clearly separate the input text from the expected label. These markers help the model recognise where the example begins and ends, and what part of the text it needs to predict.
  - `<start_of_turn>`: Marks the beginning of a training turn.
  - `<end_of_turn>`: Marks the end of a training turn.
  - `user`: Indicates a human message, i.e. text to be classified.
  - `model`: Indicates the model's response, i.e. category/class for that input.
  - `\n`: Adds line breaks for readability.

These markers help the model learn structured conversational patterns and distinguish between prompts and responses.

**Practical tips:**

- **Consistency:** Keep label spelling and case identical across all records.
- **Balance:** Aim for a roughly equal number of examples for each category to avoid bias.
- **Explicit class list:** Depending on your domain, the number of target classes, and how you intend to prompt the model, it can be helpful to include the complete list of available classes directly in the prompt (for example: 'Classify the African cultural item into one of these categories: Food, Music, Art'). This makes the task explicit and reduces ambiguity as the model is aware of all valid options.

```python
# Special tokens for start/end of a turn.
SOT = "<start_of_turn>"
EOT = "<end_of_turn>"


def format_io_as_turns(
    input_text: str, label_text: str, sot: str = SOT, eot: str = EOT
) -> tuple[str, str]:
    """
    Format a single input/output pair for LoRA training
    by wrapping it with <start_of_turn> and <end_of_turn> markers.

    Args:
      input_text: The full user prompt text. This typically includes
                  the instruction and the item content, e.g.,
                  "Classify the following item by category: <name>.".
      label_text: The correct category label for the item.
      sot: The token used to indicate the start of a turn.
           Defaults to "<start_of_turn>".
      eot: The token used to indicate the end of a turn.
           Defaults to "<end_of_turn>".

    Returns:
      formatted_q: The user turn string, e.g.
                   "<start_of_turn>user\n<prompt><end_of_turn>".
      formatted_a: the model turn string, e.g.
                   "<start_of_turn>model\n<label><end_of_turn>".
    """

    formatted_q = f"{sot}user\n{input_text}{eot}\n"
    formatted_a = f"{sot}model\n{label_text}{eot}"
    return formatted_q, formatted_a


# Load the dataset created by the instruction-based transformation step.
# Each line is a JSON object: {"input": "...", "output": "..."}

inputs = []  # Model prompts.
targets = []  # Expected responses.

for example in train_data:
    # Format each record into a user/model dialogue pair.
    q, a = format_io_as_turns(example["input"], example["output"])
    inputs.append(q)
    targets.append(a)

# Show the first set of input and output.
print(inputs[0])
print(targets[0])

# The Gemma3 Keras models require the data as dictionaries with keys
# "prompts" for the inputs and "responses" for the outputs.
data = {"prompts": inputs, "responses": targets}
```

#### Your Implementation


In [ ]:
# Add your code to add special delimiter tokens here.


### Load Gemma using Keras
Once your data is ready, you now need to fine-tune your Gemma model using LoRA,  as you familiarized yourself with in Course 05 Fine-tune Your Model. You will train only a small number of additional parameters whilst keeping the base model frozen. You may find additional information in [Google's *Fine-tune Gemma in Keras using LoRA* documentation](https://ai.google.dev/gemma/docs/core/lora_tuning).

------
> 💡 **Tip: Curious about model size and performance tradeoffs?**
>
> You can experiment with the larger **Gemma3-4B** model to compare accuracy and output quality against the 1B version.  
> Keep in mind that Colab's default **T4 GPU** may not have enough memory to fine-tune the 4B model in full precision.  
> To make this feasible, apply the **optimization techniques** introduced in Course 07 Accelerate Your Model, such as mixed-precision inference, parameter-efficient fine-tuning, or low-bit quantisation, to reduce memory usage and stay within hardware limits.
>
------
More information about the Gemma3 models can be found in [Google's *Gemma3 model overview* document](https://ai.google.dev/gemma/docs/core).


#### Worked example: Load Gemma3



```python
# Load the Gemma3-1B Keras model.
model = keras_nlp.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()
```

#### Your implementation

In [ ]:
keras.utils.set_random_seed(812)  # Replicate randomness in Keras.

# Add your code for loading a Gemma3 model here.


### Evaluate the base model's performance

Before fine-tuning, this is a good moment to **test the base (foundation) Gemma model** with a small set of example questions or prompts.
You can **reuse these same questions after fine-tuning** to see how LoRA adaptation changes the model's answers or classification behavior, giving you a direct, side-by-side comparison of performance.
For efficiency, **load the model only once** and keep it in memory while you carry out both the pre-fine-tuning tests and the later evaluation.
On Google Colab, where the default **T4 GPU has limited memory**, repeatedly loading different versions of the model can easily exceed available resources and trigger out-of-memory errors.

#### Worked example: Select spot-checking examples

One effective method for getting a sense of how well the base model does, is to sample a few examples from your validation set and pass them through the model. The following code samples five examples from the validation dataset.

```python
# Randomly sample 5 examples from the validation dataset.
manual_spot_checking_examples = random.sample(val_data, k=5)
```

#### Your Implementation


In [ ]:
# Add your code here for sampling validation examples here.


Now, ask the base model to classify each item using a simple instruction prompt. This will give you an idea of how the model performs *before* any fine-tuning. For more information on running Gemma with Keras see [Google's *Run Gemma with Keras* documentation](https://ai.google.dev/gemma/docs/core/keras_inference).

#### Worked example: Classify examples using base model



```python
for example in manual_spot_checking_examples:
    formatted_q, _ = format_io_as_turns(example["input"], example["output"])
    print(f"Prompt: {example['input']}")
    # max_length is set to 256 as the prompt is included in the token count.
    print(model.generate(formatted_q, max_length=256))
    print(f"Expected category: {example['output']}")
    print("-" * 50)
```


#### Your Implementation


In [ ]:
# Add your code for generating responses with the base model here.


------
> ⚠️ **Understanding base model limitations**
>
> You will likely notice that the base Gemma3-1B model produces inconsistent or inaccurate responses when asked to classify these items. This behavior is expected, and why is explained below:
>
> **Base models vs instruction-tuned models**:
>
> The Gemma3-1B model used here is a **base (pretrained) model**, not an **instruction-tuned** variant. Base models are trained solely on **next-token prediction**, learning to predict the most likely next word given the preceding context. They have not been further trained to understand and follow human instructions.
>
> In contrast, **instruction-tuned models** (such as `gemma3_instruct_1b`) undergo additional training on instruction-response pairs, teaching them to interpret prompts as commands and respond appropriately. When you ask a base model to 'classify' something, it may simply continue generating text in a way that seems plausible given the input, rather than providing a structured classification answer.
>
> For more information about the model variants, see [*Run Gemma content generation and inferences* documentation](https://ai.google.dev/gemma/docs/run).
>
> **Why small models struggle more**:
>
> With only 1 billion parameters, the Gemma3-1B model has limited capacity to capture complex patterns and implicit task instructions from context alone. Larger models can sometimes infer task requirements from prompt phrasing, but smaller base models are more prone to producing off-topic or hallucinated responses, generating text that sounds plausible but is factually incorrect or irrelevant to the task, or even going into producing gibberish.
>
> **Why use a base model for fine-tuning**:
>
> Despite these limitations, base models are often preferred for fine-tuning. Because they have not been optimized for a specific instruction-following format, they offer a more neutral starting point. This allows LoRA fine-tuning to more effectively steer the model's behavior towards your specific classification task, without having to override patterns learned during instruction tuning.
>
> By comparing the base model's outputs before and after fine-tuning, you will see a clear demonstration of how LoRA adaptation transforms an unreliable base model into a capable classifier.
------


### Activity 3: Activate and run LoRA



------
> 💻 **Your task:**
>
> Set up your hyperparameters and fine-tune your Gemma model using LoRA:
>
> 1. **Experiment with LoRA rank**: Start with `rank=4` (faster), `rank=6` (medium) or try `rank=8` (potentially better quality, but requires more memory):
>    ```python
>    lora_rank = 6
>    model.backbone.enable_lora(rank=lora_rank)
>    ```
>
> 2. **Adjust training parameters**:
>    - **Learning rate**: A good value to start with is `5e-5` but depending on your dataset size and how diverse your data is, you likely also want to experiment with different learning rates.
>    - **Number of epochs**: For the worked example, it is set to 3. Dialogue-based models may typically require more epochs than other type of models such as classification because they learn free-form response generation and conversational structure rather than closed-set label prediction. Training progress should be monitored to avoid overfitting.
>    - **Batch size**: Limited memory on T4 GPUs might prevent you from increasing this parameter. If you have access to a GPU with more memory, you can raise this to speed up and stabilize training.
>    - **Sequence length**: Adjust based on your text length (default: 450). You might encounter out of memory problems on a T4 GPU if sequence length is too long during fine-tuning.
>
> 3. **Monitor training progress**:
>    - Watch the loss decrease over epochs.
>    - Check sparse categorical accuracy improvement.
>    - Stop early if the model starts overfitting.
>    - You may also implement code that prints a prompt at every step.
>
> 4. **Iterate as needed**: If results are not satisfactory:
>    - Try different hyperparameters.
>    - Increase/decrease epochs.
>    - Adjust the LoRA rank.
>
------


#### Worked example: Set up hyperparameters


```python
# Set the LoRA rank.
lora_rank = 6
# Set the learning rate.
learning_rate = 5e-5
# Determine the number of epochs.
num_epochs = 3
# Set the batch size.
batch_size = 1
# Set the maximum length.
sequence_length = 450

# Activate LoRA.
model.backbone.enable_lora(rank=lora_rank)

# Check learning rate.
print(model.optimizer.learning_rate)
```

#### Your implementation

In [ ]:
# Add your code to set the hyperparameters and enable LoRA training.


-----
> **ℹ️ Info: Understanding LoRA hyperparameters**
>
> The following parameters control how LoRA fine-tuning adapts the model to your classification task:
>
> **`lora_rank`** (integer, e.g., 4, 8, 16):
> The rank of the low-rank matrices added to the model's attention layers. Higher ranks capture more complex patterns but increase memory usage and training time. For small datasets or simple classification tasks, `rank=4` is often sufficient. For larger datasets or tasks requiring nuanced understanding, try `rank=8` or higher.
>
> **`learning_rate`** (float, e.g., 1e-5, 5e-5):
> Controls how much the model weights are adjusted during each training step. Higher rates learn faster but risk overshooting optimal values. Lower rates are more stable but train slower. Start with `2e-5` or `5e-5` for LoRA fine-tuning. If the model fails to learn (high loss), try increasing slightly. If training is unstable (loss fluctuates), reduce the rate.
>
> **`num_epochs`** (integer, e.g., 2, 5, 10):
> The number of complete passes through the training dataset. More epochs allow the model to learn more thoroughly, but too many can cause **overfitting** (memorizing rather than generalizing). Start with 2–3 epochs for small datasets and increase if validation performance is still improving (monitor the `loss` and `sparse_categorical_accuracy` values). Also monitor the gap between seen and unseen data performance later on during the evaluation of the model.
>
> **`batch_size`** (integer, e.g., 1, 4, 8):
> The number of training examples processed together before updating weights. Larger batches provide more stable gradient estimates but require more GPU memory. On T4 GPUs with limited memory, `batch_size=1` is often necessary for large models like Gemma. If memory permits, increasing batch size can speed up training.
>
> **`sequence_length`** (integer, e.g., 256, 450, 512):
> The maximum number of tokens the model processes per input. Must be long enough to contain your full prompts and expected responses. Longer sequences require more memory. Analyze your training data to determine appropriate length and set it slightly above your longest expected input/output combination.
>
-----


### Fine-tuning
It is now time to perform fine-tuning with LoRA. You can leave the hyperparameters as default or change them. You may wish to reduce the number of epochs. In the first run watch the loss and the sparse categorical accuracy for each epoch and adjust accordingly to achieve the desired accuracy without overfitting the model.

The time required to complete LoRA fine-tuning depends on several factors, including the size of the dataset, the length of the prompts, and hyperparameters such as the number of epochs and LoRA rank. Depending on these variables, the process can take anywhere from a few minutes to several hours. For the default example dataset, the estimated fine-tuning time is about 15 minutes on a T4 GPU.

#### Worked example: Run fine-tuning



```python
model.optimizer.learning_rate = learning_rate
model.preprocessor.sequence_length = sequence_length

# Using a for loop provides better control over each iteration if you want to
# add custom logic, but otherwise identical to setting epochs=num_epochs.
for i in jnp.arange(num_epochs):
    print("\n\nEpoch:" + str(i + 1) + "\n")
    model.fit(data, epochs=1, batch_size=batch_size, verbose=1)
    # Add code to print model generations for one or two prompts after every
    # epoch here.
```

#### Your implementation

In [ ]:
# Add your code to fine-tune the model.


Ideally, you now have a fine-tuned Gemma model trained with LoRA for your classification task. The next step is to evaluate how well the classifier performs by testing it on real examples and conducting systematic evaluation.

## Activity 4: Model evaluation


------
> 💻 **Your task:**
>
> Evaluate your fine-tuned classifier through a combination of manual spot checks and quantitative metrics. You will assess performance on both **seen data** (training examples) and **unseen data** (test examples) to understand how well your model generalizes. Now is the time to use your test set. If you use a third-party dataset with existing splits, it is good practice to keep those original splits to maintain comparability with other work.
>
> 1. **Compare base model vs. fine-tuned model**
>    - Generate examples using the examples in `manual_spot_checking_examples` that you used earlier for testing the base model. Observe the difference in model behavior:
>       - Does the fine-tuned model now produce concise, single-word category labels?
>       - How does the structured output compare to the base model's off-topic or rambling responses?
>
> 2. **Run systematic evaluation**:
>    - Predict the classes for all examples in the training, validation, and test data.
>    - Compute accuracy, precision, and recall and compare them across the training set and the validation and test sets to assess generalization.
>
> 3. **Assess model performance**:
>    - **Accuracy**: How often does the model predict correctly?
>    - **Generalization**: Is there a large gap between seen and unseen performance?
>    - **Per-class patterns**: Which categories perform well or poorly?
>    - **Format consistency**: Does the model always return single-category labels?
>    - **Edge cases**: How does it perform on ambiguous or difficult examples?
>
> 4. **Document your findings and iterate**: Record what works well and what needs improvement for future iterations. You may want to refine the hyperparameters so that you get better results on the validation set and then see whether these improvements also translate to the held-out test set.
>
>
------


#### Comparing base model vs fine-tuned model outputs

Now that the model has been fine-tuned with LoRA, you can test it using the **same items** you used earlier to evaluate the base model. This allows for a direct, side-by-side comparison of how fine-tuning has affected the model's classification behavior.

In the next code cell, implement code to classify the same set of items using the fine-tuned model. Compare the outputs with what you observed earlier from the base model:

- **Base model (before fine-tuning)**: This model likely produced inconsistent, off-topic, or hallucinated responses because it was trained only on next-token prediction and has not learned the intricacies of the classification task.

- **Fine-tuned model (after LoRA)**: This model should produce more accurate category labels that match the expected classifications, demonstrating how LoRA adaptation has taught the model to follow the classification instruction.


#### Worked example: Manual spot checking of fine-tuned model


```python
for example in manual_spot_checking_examples:
    formatted_q, _ = format_io_as_turns(example["input"], example["output"])
    print(f"Prompt: {example['input']}")
    # max_length is set to 256 as the prompt is included in the token count.
    print(model.generate(formatted_q, max_length=256))
    print(f"Expected category: {example['output']}")
    print("-" * 50)
```

#### Your Implementation


In [ ]:
# Add your code to run manual spot-checking of fine-tuned model here.


When making predictions, **the model ideally will now follow the expected response format in most cases**. This means outputting a single classification word rather than the rambling, off-topic text produced by the base model earlier. This structured behavior is a direct result of LoRA fine-tuning, which taught the model:

1. **What task to perform**: Classify the input into one of the learned categories.
2. **How to respond**: Produce a concise, single-word answer.

The `<start_of_turn>` and `<end_of_turn>` tokens play a key role here. These special delimiters clearly demarcate user input from model output, helping the fine-tuned model interpret when to 'listen' and when to 'respond'. This turn-based structure, standard in instruction-tuned models, guides the model toward appropriate, on-task behavior.

Compare these results to the base model's outputs from earlier in the notebook: the contrast illustrates how fine-tuning transforms a base autocomplete model into a task-specific classifier.

### Generate and save predictions for full evaluation

Implement code to generate predictions for **all entries** in both datasets and save them to separate CSV files:

- **`predictions_train.csv`**: Predictions on the training data. High accuracy here confirms the model learned the training examples.
- **`predictions_val.csv`**: Predictions on validation data. This measures generalization performance that you can try to improve by adapting hyperparameters, modifying your training data or using different foundation models.
- **`predictions_test.csv`**: Predictions on test data. This can be used to measure generalization performance once the model has been fully optimized.

This comprehensive evaluation allows you to compare performance between seen and unseen data, helping identify potential overfitting.

#### Worked example: Generate predictions for all data


The code in the worked example defines a function that computes and saves predictions for an entire dataset. It then uses this function to compute predictions for training, validation, and test data.

```python
def generate_all_predictions(
    data: Dict[str,str],
    model: keras_nlp.models.Gemma3CausalLM,
    output_file: str,
) -> pd.DataFrame:
    """
    Generate predictions for all entries in a dataset and save to CSV.

    Args:
      data: Dictionary of input and output for each prediction.
      model: The model used to generate predictions.
      output_file: Path to save the predictions CSV.

    Returns:
      DataFrame with all predictions added.
    """

    df_predictions = pd.DataFrame(data)

    # Generate model predictions.
    print(f"Generating predictions for {len(df_predictions)} entries.")
    for idx, row in tqdm(df_predictions.iterrows(), total=len(df_predictions)):
        prompt, _ = format_io_as_turns(row["input"], row["output"])
        model_output = model.generate(prompt, max_length=512)
        df_predictions.at[idx, "model_output"] = model_output

    # Save to CSV file.
    df_predictions.to_csv(output_file, index=False, encoding="utf-8")
    print(f"Saved {len(df_predictions)} predictions to {output_file}")

    return df_predictions


# Generate predictions for training data.
print("=" * 50)
print(" Generating predictions for training data:")
print("=" * 50)
df_predictions_train = generate_all_predictions(
    data=train_data,
    output_file="predictions_train.csv",
    model=model,
)

# Generate predictions for validation data.
print("=" * 50)
print(" Generating predictions for validation data:")
print("=" * 50)
df_predictions_val = generate_all_predictions(
    data=val_data,
    output_file="predictions_val.csv",
    model=model,
)

# Generate predictions for test data.
print("=" * 50)
print(" Generating predictions for test data:")
print("=" * 50)
df_predictions_test = generate_all_predictions(
    data=test_data,
    output_file="predictions_test.csv",
    model=model,
)
```

#### Your Implementation


In [ ]:
# Add your code here to generate and save predictions for all datasets.


### Load and review saved predictions

The following helper function loads predictions from a saved CSV file and displays them in a readable format. This allows you to review the model's outputs without regenerating predictions, which is useful for iterative analysis or sharing results with others.

In [ ]:
def load_and_display_predictions(filepath: str, title: str) -> pd.DataFrame:
    """
    Load predictions from a CSV file and display summary information.

    Args:
      filepath: Path to the predictions CSV file.
      title: Title to display above the results.

    Returns:
      DataFrame with loaded predictions.
    """
    df = pd.read_csv(filepath)

    print(f"\n{'=' * 50}")
    print(f" {title}")
    print(f"{'=' * 50}")
    print(f"Total predictions: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    # Display the predictions.
    display_cols = [
        c for c in ["input", "output", "model_output"]
        if c in df.columns
    ]
    display(df[display_cols])

    return df

### Display all predictions

Implement code to load and display the complete prediction results for both training and validation data, using the `load_and_display_predictions` function that you defined in the previous cell. This provides a comprehensive view of the model's classifications across all items, allowing you to manually inspect outputs and identify patterns or errors before computing quantitative metrics.

#### Worked example: Load and display predictions



```python
# Load and display training data predictions
df_pred_seen = load_and_display_predictions(
    "predictions_train.csv",
    "Training data classification"
)

# Load and display validation data predictions
df_pred_unseen = load_and_display_predictions(
    "predictions_val.csv",
    "Validation data classification"
)
```


#### Your Implementation


In [ ]:
# Add your code to load and display predictions here.


### Evaluating model predictions: spot-checking guide

After running your evaluation code above, you should see a table with several columns, including `output` and  `model_output`. You can evaluate your model's performance by comparing these two key columns.

#### Understanding the model output format

The `model_output` column contains the raw response from your fine-tuned Gemma model. In the worked example, you would see something like the following output:

```
Classify the following item by category: Jollof Rice.<end_of_turn>
<start_of_turn>model
Food<end_of_turn><end_of_turn>
```

**Key components to identify:**
- **Input prompt**: `"Classify the following item by category: [name]."`.
- **Model prediction**: `"Food"` (this is what you need to extract).
- **Special tokens**: `<start_of_turn>`, `<end_of_turn>` (formatting tokens that can be ignored).

#### Step-by-step evaluation process

**1. Extract the model's prediction**
From the `model_output` column, identify the text between `<start_of_turn>model` and the first `<end_of_turn>`. In the example above, the model predicted: `Food`.

**2. Compare with the true label**
Look at the `output` column for the same row. This contains the correct classification.

**3. Determine accuracy**
- ✅ **Correct prediction**: If the model's prediction matches the `output` value exactly.
- ❌ **Incorrect prediction**: If they don't match.

#### Example

| name | output | model_output | Evaluation |
|------|----------|--------------|------------|
| Jollof Rice | Food | ...Food&lt;/end_of_turn&gt; | ✅ CORRECT |
| Afrobeat | Music | ...Food&lt;/end_of_turn&gt; | ❌ INCORRECT |

#### What to look for during spot-checking

**✅ Good signs:**
- Model predictions match the `output` column consistently.
- Predictions are single, clear category labels (not long explanations).
- Model handles diverse examples within each category.

**⚠️ Warning signs:**
- Frequent mismatches between prediction and true category.
- Model outputs contain unexpected text or formatting.
- Systematic bias toward certain categories.
- Inconsistent predictions for similar cultural items.

#### Systematic evaluation tips

1. **Count accuracy**: Track how many predictions match the true labels out of your sample size.
2. **Identify patterns**: Note which categories are most/least accurate.
3. **Check edge cases**: Pay attention to ambiguous cultural items that could belong to multiple categories.
4. **Validate formatting**: Ensure predictions are properly extracted from the model output.

This manual evaluation process helps you understand your model's strengths and weaknesses before deploying it in real-world applications.


### Automate your evaluation using quantitative metrics

Write code to to automatically compare your **ground-truth column** (for example, the `output` column) with the **model's predicted class** extracted from the text between `<start_of_turn>model` and `<end_of_turn>` in the `model_output` field.

A reminder on some key quantitative classification metrics to work with:

* **Accuracy:** The ratio of correctly predicted instances to the total number of predictions. It provides an overall measure of performance but can be misleading in the presence of class imbalance.

* **Precision:** The proportion of positive predictions that are actually correct. It quantifies the model's reliability in its positive predictions, particularly relevant when false positives incur high cost.

* **Recall:** This metric measures the model's ability to capture all relevant instances within a class, especially critical when missing positive cases is undesirable.

In the context of a fine-tuned LLM used for classification, you can compute predictions from the model output strings and then apply these metrics to quantify performance across your dataset. Given this is a text-generation-based classifier rather than a traditional model with fixed output heads, treat label-generation accuracy as the basis for these calculations. For more information on how to calculate and interpret these metrics, take a look at the article [*Classification: Accuracy, recall, precision, and related metrics* crash course](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall).

These metrics provide a quantitative view but may not be able to capture all aspects of a classifier's behavior using LLMs. Sometimes, the model may also fail to follow the expected format, so your code should handle such cases gracefully. Human review and contextual checks are important complementary analyses.


#### Worked example: Extract predictions and compute automatic metrics



The following code demonstrates how to compute quantitative evaluation metrics using the CSV files saved earlier. It parses the `model_output` column to extract the predicted class and compares it with the ground-truth `output` column.

The code:
1. Defines a reusable `evaluate_predictions` function that extracts predicted labels and calculates metrics.
2. Runs evaluation on the training data, the validation data, and the test data.

```python
def extract_prediction(model_output: str) -> str:
    """Extract the predicted class from the model output string.

    Parses the raw model output to find the classification label between
    the last <start_of_turn>model and the following <end_of_turn> tokens.

    Args:
      model_output: The raw string output from the model, containing
                    special turn tokens and the predicted class label.

    Returns:
      str: The extracted class label (e.g., 'Food', 'Music'), or
           "Unknown" if no valid prediction could be extracted.
    """
    pattern = r"<start_of_turn>model\s*([^<]*)<end_of_turn>"
    matches = re.findall(pattern, model_output)

    if matches:
        return matches[-1].strip()
    return "Unknown"


def evaluate_predictions(filepath: str, title: str) -> dict:
    """Load predictions from CSV and compute evaluation metrics.

    Args:
      filepath: Path to the predictions CSV file.
      title: Title to display above the results.

    Returns:
      dict: Dictionary containing accuracy, precision, recall, and
            the evaluated DataFrame.
    """
    # Load predictions.
    df_eval = pd.read_csv(filepath)

    # Extract predicted labels from model output.
    df_eval["predicted"] = df_eval["model_output"].apply(extract_prediction)

    print(f"\n{'=' * 50}")
    print(f" {title}")
    print(f"{'=' * 50}")

    # Display sample of extracted predictions.
    print("\nSample predictions:")
    display(df_eval[["output", "predicted"]].head(5))

    # Calculate metrics.
    y_true = df_eval["output"]
    y_pred = df_eval["predicted"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\nAccuracy: {accuracy:.2%}")
    print(f"Precision (weighted): {precision:.2%}")
    print(f"Recall (weighted): {recall:.2%}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "df": df_eval,
    }


# Evaluate training examples.
results_train = evaluate_predictions(
    "predictions_train.csv", "Training examples:"
)

# Evaluate validation examples.
results_val = evaluate_predictions(
    "predictions_val.csv", "Validation examples:"
)

# Evaluate test examples.
results_test = evaluate_predictions(
    "predictions_test.csv", "Test examples:"
)

# Summary comparison.
print("\n" + "=" * 50)
print(" Summary: Seen vs unseen performance")
print("=" * 50)
print(
    f"{'Metric':<20} {'Train':<15} {'Val':<15} {'Test':<15}"
)
print("-" * 50)
print(
    f"{'Accuracy':<20} {results_train['accuracy']:.2%}{'':<7} "
    f"{results_val['accuracy']:.2%}{'':<7} "
    f"{results_test['accuracy']:.2%}{'':<7} "
)
print(
    f"{'Precision':<20} {results_train['precision']:.2%}{'':<7} "
    f"{results_val['precision']:.2%}{'':<7} "
    f"{results_test['precision']:.2%}{'':<7} "
)
print(
    f"{'Recall':<20} {results_train['recall']:.2%}{'':<7} "
    f"{results_val['recall']:.2%}{'':<7} "
    f"{results_test['recall']:.2%}{'':<7} "
)
```


#### Your Implementation


In [ ]:
# Add your code to run quantitative evaluation here.


### Interpreting the quantitative metrics results

The results should ideally show a moderate performance gap between seen (training) and unseen (validation and test) data. This would suggest that the model has learned meaningful classification patterns and exhibits reasonable generalization capability.

**Understanding the training vs. validation/test comparison:**

- **High training accuracy, lower validation/test accuracy**: This is a common pattern. The model naturally performs better on data it was trained on. A moderate gap is expected and acceptable.

- **A large gap**: This may indicate **overfitting**, where the model learned noise in the training examples rather than learning generalizable patterns. Consider reducing epochs, increasing training data diversity, or applying regularization to prevent overfitting.

- **Similar performance on both**: Indicates good generalization. The model has learned underlying patterns rather than overfitting on noise in the training data.

**Key considerations:**

- **Training data volume matters**: More training data typically improves generalization, particularly for items the base model is less familiar with.

- **Consider additional training epochs**: If seen accuracy is low, additional epochs may help. However, watch for an increasing gap with unseen data, which signals overfitting.

- **Fine-tuning is not a knowledge database**: Fine-tuning teaches the model *how* to respond to classification prompts, not encyclopaedic knowledge about every possible input. The model can still produce incorrect classifications for items outside its learned patterns.

For production applications, prioritize **unseen data performance** as the true measure of model quality, and continue to expand your training dataset with diverse, high-quality examples.


You have now completed the technical implementation and evaluation of your text classifier. Through a combination of manual spot checks, quantitative metrics, and comparative analysis of seen versus unseen examples, you've assessed how well your fine-tuned model performs on real classification tasks.

Now take some time to reflect on your learning journey and document your insights for future improvements.

## Activity 5: Iteration

One of the key parts of building any machine learning system is experimenting with different hyperparameters, datasets, models, or other variations of the pipeline that you just implemented. Even the most experienced machine learning researchers and engineers rarely get all of these things right without systematic experiments.

If you identified any systematic issues or are otherwise not yet satisfied with the performance of your classifier, think how you may be able to improve your system. You can then go back to individual steps, make changes and repeat the training and evaluation of your model to determine whether the changes are effective.

To do these iterations most systematically, it is best if you do not change too many things at once, so that you learn which components affect your classifier the most. Furthermore, make sure to keep a log of your experiments somewhere in which you detail the settings of each training run, so that you know at the end what worked best. That way, you can progressively optimize your machine learning model until you are satisfied with its performance.

## Activity 6: Reflection


------
> 💻 **Your task:**
>
> **Technical and project reflection:**
>
> Document your learning and plan improvements.
>
> - **What worked well:** Reflect on successful aspects of your classifier
>
> - **Challenges encountered:** What difficulties did you face and how did you solve them?
>
> - **Data quality observations:** How was the quality of your training data? What would you do differently?
>
> - **Model performance:** How well did your classifier perform? What are its strengths and weaknesses?
>
> - **Classification-specific insights:** What did you learn about text classification? Which classes were hardest to distinguish?
> ------
> **Learning journey reflection:**
>
> Take a few minutes to think about your own learning journey in this course. Use these prompts to guide a short written reflection.
>
>  - **What aspects of the course helped you understand large language models and LoRA fine-tuning most effectively?** Which study habits or tools did you find especially valuable for your progress?
>
>  - **How did your thinking about machine-learning workflows change as you moved from data collection to fine-tuning and evaluation?** What surprised you or challenged your assumptions?
>
>  - **Looking beyond this specific project, how will you apply these skills and insights when approaching new ML tasks or unfamiliar datasets in the future?**
>
>  - **What attitudes, ethics, or professional values do you believe are essential for a responsible ML practitioner?** How will you cultivate these qualities as you continue to develop as a machine-learning engineer?
------

No example solution is provided for the reflection activity. This exercise is designed to be completed independently by you, drawing on your own experiences, challenges, and insights from working through this capstone project. Your reflections are personal and unique to your learning journey.

### Future improvements

- _[List 3-5 ways to improve your classifier]_
-
-
-
-

### Course connections

_[Which course concepts were most important for this project?]_

## Summary

Congratulations, you have completed the Text Classification Capstone project.

This capstone demonstrated a complete machine learning workflow from conception to evaluation, highlighting the importance of careful data curation, parameter-efficient training methods, and evaluation practices in building reliable AI systems.